# Extended Temperature Timeseries (2026 read-outs)

Working versions of the published temperature-timeseries figures, extended with the
July 2026 read-outs. The paper figures themselves are produced by notebooks 3 and 5 and
are **not** touched here — everything written by this notebook goes to
`products/figures/icetemp_results/extended/`.

**Part 1 — Splice the 2026 read-outs onto the existing records**
- Tynitag NTC loggers → `NTC_tynitag/temperature_data/full_timeseries/`
- Geoprecision chains → `thermistor_chains/temperature_data/full_timeseries/`

**Part 2 — Extended figures**
- Depth-coloured Tynitag timeseries, all six sites (extended fig03)
- Geoprecision heatmap mosaics for Alphubel, Chessjen and Hohsaas (extended figS14–S16)

**What the 2026 read-out covers**

| Site | Tynitag | Geoprecision |
|---|---|---|
| Alphubel (AH) | AH1TT, AH2TT, AH3TT → 30 Jul 2026 | AH1G, AH2G, AH3G → 30 Jul 2026 |
| Chessjen (CJ) | CJ1TT, CJ2TT → 10 Jul 2026 | CJ1G, CJ2G → 10 Jul 2026 |
| Hohsaas (HS) | HS1TT, HS2TT → 29 Jul 2026 | HS1G, HS2G, HS3G → 29 Jul 2026 |
| Sex Rouge (SR) | SR1TT, SR2TT → 27 Jul 2026 | — |
| Tortin (GT) | GT2TT → 28 Jul 2026 (GT1TT not read out) | — |
| Corvatsch (CV) | not read out — record ends 5 Sep 2025 | — |

---

## Part 1 — Splice the 2026 read-outs

## 1. Imports and paths

In [ ]:
from config import ICETEMP_ROOT
%matplotlib inline
import os
import shutil
import sys
import tempfile
import time

import matplotlib as mpl
import matplotlib.cm as mcm
import matplotlib.colors as mcolors
import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.colorbar import ColorbarBase
from matplotlib.gridspec import GridSpec
from matplotlib.lines import Line2D
from scipy.optimize import curve_fit

mpl.rcParams['figure.dpi'] = 100

project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from src.thermistor_processing import ThermistorData, splice_timeseries
from src.thermistor_plotting import plot_chain_temperature_heatmap, mosaic_chain_heatmaps

In [ ]:
# Tynitag NTC
TT_ROOT = os.path.join(ICETEMP_ROOT, "NTC_tynitag", "temperature_data")
TT_FULL = os.path.join(TT_ROOT, "full_timeseries") + "/"
TT_2026 = os.path.join(TT_ROOT, "2026") + "/"

# Geoprecision chains
GP_ROOT = os.path.join(ICETEMP_ROOT, "thermistor_chains", "temperature_data") + "/"
GP_FULL = os.path.join(GP_ROOT, "full_timeseries") + "/"

# Borehole settings and calibration
BH_SET = os.path.join(project_root, "data", "borehole_settings") + "/"
OFFSETS_TT = pd.read_csv(os.path.join(ICETEMP_ROOT, "NTC_tynitag", "calibration_data",
                                      "all_logger_offsets.csv"))
OFFSETS_GP = pd.read_csv(os.path.join(project_root, "data", "calibration",
                                      "corrected_chain_offsets.csv"), index_col="chain")

# Output — deliberately separate from figures/paper and figures/supplement
OUT_DIR = os.path.join(project_root, "products", "figures", "icetemp_results", "extended") + "/"
os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(GP_FULL, exist_ok=True)


def savefig(fig, path, **kw):
    """Render to local disk, then copy into place.

    OUT_DIR sits in iCloud Drive, where matplotlib's incremental writes raise
    TimeoutError [Errno 60] whenever the sync queue is busy. Rendering locally
    turns dozens of small writes into a single copy, retried with backoff, and
    the copy-then-replace means a stalled write never truncates an existing figure.
    """
    fd, local = tempfile.mkstemp(suffix=os.path.splitext(path)[1] or ".pdf")
    os.close(fd)
    part = path + ".part"
    try:
        fig.savefig(local, **kw)
        last_exc = None
        for attempt in range(5):
            try:
                shutil.copyfile(local, part)
                os.replace(part, path)
                return path
            except (TimeoutError, OSError) as exc:
                last_exc = exc
                delay = 5 * 2 ** attempt
                print(f"  {os.path.basename(path)}: {exc} — retry {attempt + 1}/5 in {delay}s")
                time.sleep(delay)
        fallback = os.path.join(tempfile.gettempdir(), os.path.basename(path))
        shutil.copyfile(local, fallback)
        raise RuntimeError(f"could not write {path} ({last_exc}). "
                           f"Figure kept at {fallback} — copy it over once iCloud settles.")
    finally:
        for tmp in (local, part):
            if os.path.exists(tmp):
                try:
                    os.remove(tmp)
                except OSError:
                    pass

## 2. Splice the Tynitag records

Each 2026 CSV starts at the timestamp where the previous read-out ended, so the records
concatenate without a gap. The 0-degree ice-bath offsets are applied to the new segment
**only where the existing series already carries them** — otherwise the splice would
introduce a step at the join.

> Two records are the exception: `AH1TT` (former BH9) and `CT1TT`/`CV1TT` (former BH11) were
> copied into `full_timeseries/` as raw logger exports, so no ice-bath offset was ever applied
> to them (logger 9: black 0.608 °C, white 0.299 °C; logger 11: black 0.340 °C, white 0.192 °C).
> `APPLY_MISSING_OFFSETS = False` reproduces the published processing; set it to `True` to
> correct both segments consistently — note this shifts AH1TT away from the values in fig03.

In [ ]:
APPLY_MISSING_OFFSETS = False

# (label, existing series, existing series carries offsets, 2026 read-out, logger id, output stem)
TT_JOBS = [
    ("SR1TT", "SR1TT_20240806_20250724_spliced.csv", True,  "SR1TT_20260727.csv", "1",  "SR1TT_20240806_20260727_spliced"),
    ("SR2TT", "SR2TT_20240806_20250724_spliced.csv", True,  "SR2TT_20260727.csv", "2",  "SR2TT_20240806_20260727_spliced"),
    ("GT2TT", "GT2TT_20240807_20250723_spliced.csv", True,  "GT2TT_20260728.csv", "4",  "GT2TT_20240807_20260728_spliced"),
    ("HS1TT", "HS1TT_20240808_20250927_spliced.csv", True,  "HS1TT_20260729.csv", "5",  "HS1TT_20240808_20260729_spliced"),
    ("HS2TT", "HS2TT_20240808_20250927_spliced.csv", True,  "HS2TT_20260729.csv", "6",  "HS2TT_20240808_20260729_spliced"),
    ("CJ1TT", "CJ1TT_20240809_20250808_spliced.csv", True,  "CJ1TT_20260710.csv", "7",  "CJ1TT_20240809_20260710_spliced"),
    ("CJ2TT", "CJ2TT_20240809_20250808_spliced.csv", True,  "CJ2TT_20260710.csv", "8",  "CJ2TT_20240809_20260710_spliced"),
    ("AH1TT", "AH1TT_20240821_20250916.csv",         False, "AH1TT_20260730.csv", "9",  "AH1TT_20240821_20260730_spliced"),
    ("AH2TT", "AH2TT_20240821_20250916_spliced.csv", True,  "AH2TT_20260730.csv", "10", "AH2TT_20240821_20260730_spliced"),
    ("AH3TT", "AH3TT_20250806_20250916.csv",         False, "AH3TT_20260730.csv", "13", "AH3TT_20250806_20260730_spliced"),
]

for label, old_name, old_has_offsets, new_name, logger_id, stem in TT_JOBS:
    old_td = ThermistorData(TT_FULL + old_name, ",")
    new_td = ThermistorData(TT_2026 + new_name, ",")

    if old_has_offsets:
        old = old_td.get_ntc_data()
        new = new_td.get_ntc_data_with_offsets(logger_id, OFFSETS_TT)
    elif APPLY_MISSING_OFFSETS:
        old = old_td.get_ntc_data_with_offsets(logger_id, OFFSETS_TT)
        new = new_td.get_ntc_data_with_offsets(logger_id, OFFSETS_TT)
    else:
        old = old_td.get_ntc_data()
        new = new_td.get_ntc_data()

    merged = splice_timeseries([old, new], time_col="TIME",
                               out_path=TT_FULL + stem + ".csv", file_stem=stem)
    print(f"{label}: {len(old):5d} + {len(new):5d} -> {len(merged):5d}   "
          f"{merged['TIME'].min():%Y-%m-%d} to {merged['TIME'].max():%Y-%m-%d}")

## 3. Splice the Geoprecision chain records

The FlexGate loggers are cleared at each read-out, so a read-out covers everything since the
previous visit. For HS1G/HS2G/HS3G/AH3G the 2026 file already spans the whole record; for
AH1G, AH2G, CJ1G and CJ2G it has to be joined to the 2025 file. The merged record is written
back in the FG2 layout that `ThermistorData.get_chain_data()` reads.

In [ ]:
# (chain, borehole, [read-outs oldest -> newest], output stem)
GP_JOBS = [
    ("A551FE", "AH1G", ["2025/A551FE/raw/A551FE_20250916075641.csv",
                        "2026/A551FE/raw/A551FE_20260730103314.csv"], "A551FE_20250714_20260730_spliced"),
    ("A55204", "AH2G", ["2025/A55204/raw/A55204_20250916082002.csv",
                        "2026/A55204/raw/A55204_20260730114344.csv"], "A55204_20250716_20260730_spliced"),
    ("A55205", "AH3G", ["2025/A55205/raw/A55205_20250916074654.csv",
                        "2026/A55205/raw/A55205_20260730104528.csv"], "A55205_20250716_20260721_spliced"),
    ("A551FD", "HS1G", ["2025/A551FD/raw/A551FD_20250927153221.csv",
                        "2026/A551FD/raw/A551FD_20260729120705.csv"], "A551FD_20250714_20260729_spliced"),
    ("A55203", "HS2G", ["2025/A55203/raw/A55203_20250927151514.csv",
                        "2026/A55203/raw/A55203_20260729111825.csv"], "A55203_20250716_20260729_spliced"),
    ("A55200", "HS3G", ["2025/A55200/raw/A55200_20250927151301.csv",
                        "2026/A55200/raw/A55200_20260729124847.csv"], "A55200_20250715_20260729_spliced"),
    ("A55201", "CJ1G", ["2025/A55201/raw/A55201_20251215143911.csv",
                        "2026/A55201/raw/A55201_20260710114409.csv"], "A55201_20250715_20260710_spliced"),
    ("A55202", "CJ2G", ["2025/A55202/raw/A55202_20251215144715.csv",
                        "2026/A55202/raw/A55202_20260710135234.csv"], "A55202_20250715_20260710_spliced"),
]

_GP_TIME_FMT = "%d.%m.%Y %H:%M:%S"


def write_chain_csv(df, out_path, logger):
    """Write a merged chain record in the FG2 layout read by get_chain_data()."""
    sensor_cols = [c for c in df.columns if c.startswith("#")]
    extra_cols = [c for c in df.columns if c not in ("NO", "TIME") and not c.startswith("#")]
    cols = sensor_cols + extra_cols
    header = ",".join(["NO", "TIME"] + [f"{c}:(unk.)" if c.startswith("#") else c for c in cols])
    with open(out_path, "w") as fh:
        fh.write(f"<LOGGER: ${logger}>\n")
        fh.write("<SPLICED: merged read-outs, notebook 10>\n")
        fh.write(header + "\n")
        for i, (_, row) in enumerate(df.iterrows(), start=1):
            vals = ["" if pd.isna(row[c]) else f"{row[c]:g}" for c in cols]
            fh.write(f"{i},{row['TIME'].strftime(_GP_TIME_FMT)}," + ",".join(vals) + "\n")


for chain, borehole, rel_paths, stem in GP_JOBS:
    frames = []
    for i, rel in enumerate(rel_paths):
        df = ThermistorData(GP_ROOT + rel, ",").get_chain_data()
        df["__chunk__"] = i          # prefer the newer read-out where they overlap
        frames.append(df)
    merged = (pd.concat(frames, ignore_index=True)
              .sort_values(["TIME", "__chunk__"])
              .drop_duplicates(subset=["TIME"], keep="last")
              .drop(columns="__chunk__")
              .sort_values("TIME")
              .reset_index(drop=True))
    merged["NO"] = range(1, len(merged) + 1)
    write_chain_csv(merged, GP_FULL + stem + ".csv", chain)
    n_sensors = len([c for c in merged.columns if c.startswith("#")])
    print(f"{borehole} ({chain}): {' + '.join(str(len(f)) for f in frames)} -> {len(merged):5d} "
          f"[{n_sensors} sensors]   {merged['TIME'].min():%Y-%m-%d} to {merged['TIME'].max():%Y-%m-%d}")

---

## Part 2 — Extended figures

## 4. Tynitag timeseries, all six sites (extended fig03)

Same construction as fig03: line colour = initial sensor depth, linestyle = borehole,
dotted after sensor failure. Two changes are forced by the longer records:

- **Shared time axis.** All panels span Aug 2024 → Aug 2026 so the differing record lengths
  are visible. Because a panel now starts before its own deployment, the 14-day
  drilling-equilibration phase is masked per site rather than by the axis limits.
- **ZAA is recomputed** on the two-year records, which changes the estimates (see below).

`GT2TT` stays excluded, as in fig03 — its white-probe ice-bath offset (5.957 °C) is unusable.
Since `GT1TT` was not read out, the Tortin panel does not extend.

In [ ]:
# ── Records: extended where a 2026 read-out exists, unchanged otherwise ───────
files = {
    "AH": [TT_FULL + "AH1TT_20240821_20260730_spliced.csv",
           TT_FULL + "AH2TT_20240821_20260730_spliced.csv"],
    "CJ": [TT_FULL + "CJ1TT_20240809_20260710_spliced.csv",
           TT_FULL + "CJ2TT_20240809_20260710_spliced.csv"],
    "HS": [TT_FULL + "HS1TT_20240808_20260729_spliced.csv",
           TT_FULL + "HS2TT_20240808_20260729_spliced.csv"],
    "SR": [TT_FULL + "SR1TT_20240806_20260727_spliced.csv",
           TT_FULL + "SR2TT_20240806_20260727_spliced.csv"],
    "GT": [TT_FULL + "GT1TT_20240807_20250723_spliced.csv",   # not read out in 2026
           TT_FULL + "GT2TT_20240807_20260728_spliced.csv"],
    "CV": [TT_FULL + "CT1TT_20240828_20250905.csv",           # not read out in 2026
           TT_FULL + "CT2TT_20240828_20250905_spliced.csv"],
}

labels = {"AH": ["AH1TT", "AH2TT"], "CJ": ["CJ1TT", "CJ2TT"], "HS": ["HS1TT", "HS2TT"],
          "SR": ["SR1TT", "SR2TT"], "GT": ["GT1TT", "GT2TT"], "CV": ["CV1TT", "CV2TT"]}

# Depths as surveyed; no 2026 survey exists yet, so Z_final is still the 2025 value.
depths_initial = {"AH": [10.0, 15.0, 9.0, 14.0],   "CJ": [6.8, 11.8, 8.5, 13.5],
                  "HS": [10.6, 15.6, 10.4, 15.4],  "SR": [10.0, 15.25, 10.0, 14.45],
                  "GT": [4.2, 9.2, 8.3, 13.3],     "CV": [2.0, 7.0, 4.3, 9.3]}
depths_current = {"AH": [9.57, 14.57, 7.81, 12.81], "CJ": [4.1, 9.1, 5.48, 10.48],
                  "HS": [8.3, 13.3, 7.5, 12.5],     "SR": [6.57, 11.82, 8.93, 13.38],
                  "GT": [1.43, 6.43, 4.87, 9.87],   "CV": [1.26, 6.26, 3.09, 8.09]}

deploy = {"AH": "2024-08-21", "CJ": "2024-08-09", "HS": "2024-08-08",
          "SR": "2024-08-06", "GT": "2024-08-07", "CV": "2024-08-28"}

top, bot = ("AH", "CJ", "HS"), ("SR", "GT", "CV")
excl = {"GT": ["GT2TT"]}
smooth_days = 3
equil_days = 14

In [ ]:
# ── Helpers (as in notebook 5) ───────────────────────────────────────────────
def smooth_df(df, days):
    if df is None or df.empty or not days:
        return df
    d = df.copy()
    d["TIME"] = pd.to_datetime(d["TIME"])
    d = d.sort_values("TIME").drop_duplicates("TIME").set_index("TIME")
    d = d.resample("6h").mean().interpolate("time", limit_direction="both")
    win = f"{float(days)}D"
    for col in ("White Probe Temperature", "Black Probe Temperature"):
        if col in d:
            d[col] = d[col].rolling(win, min_periods=1, center=True).mean()
    return d.reset_index()


def first_nan(series):
    mask = np.isnan(np.array(series))
    return int(np.argmax(mask)) if mask.any() else len(mask)


def load_bh_data(paths, smooth):
    dfs = []
    for fp in paths:
        df = ThermistorData(fp, ",", None).get_ntc_data()
        if smooth and float(smooth) > 0:
            df = smooth_df(df, smooth)
        df["TIME"] = pd.to_datetime(df["TIME"])
        dfs.append(df.sort_values("TIME"))
    return dfs

### 4.1 Zero Annual Amplitude on the two-year records

Same fit as notebook 5: an exponential amplitude decay `A(z) = A0 exp(-z/d)` through the
per-sensor seasonal amplitudes (2nd–98th percentile range), reported only where the deepest
sensor is already below 0.3 °C. A second seasonal cycle raises the measured amplitudes, so
the estimates move — and Chessjen now falls outside the reporting criterion.

In [ ]:
AMP_THRESHOLD = 0.3   # °C — skip ZAA if the deepest sensor is still above this
ZAA_SIGNAL    = 0.05  # °C — amplitude level defining ZAA
PAPER_ZAA     = {"AH": 17, "CJ": 20, "HS": 18, "SR": 23, "GT": None, "CV": None}

zaa_files  = {c: list(files[c]) for c in files}
zaa_files["GT"] = [files["GT"][0]]                       # GT2TT excluded
zaa_depths_in = {c: list(depths_current[c]) for c in depths_current}
zaa_depths_in["GT"] = [1.43, 6.43]


def sensor_amplitudes(code):
    pairs, d_idx = [], 0
    depth_list = zaa_depths_in[code]
    for fpath in zaa_files[code]:
        df = ThermistorData(fpath, delimiter=",").get_ntc_data().sort_values("TIME")
        t0, t1 = df["TIME"].min(), df["TIME"].max()
        if (t1 - t0).days >= 330:
            t0 = t0 + pd.Timedelta(days=equil_days)
        df = df[(df["TIME"] >= t0) & (df["TIME"] <= t1)]
        for col in ["White Probe Temperature", "Black Probe Temperature"]:
            if col in df.columns and d_idx < len(depth_list):
                v = df[col].dropna()
                if len(v) > 100:
                    pairs.append((depth_list[d_idx], float(np.percentile(v, 98) - np.percentile(v, 2))))
                d_idx += 1
    return sorted(pairs, key=lambda x: x[0])


def fit_zaa(pairs):
    if not pairs:
        return None, np.nan
    z = np.array([p[0] for p in pairs])
    a = np.array([p[1] for p in pairs])
    deepest_amp = a[np.argmax(z)]
    if deepest_amp >= AMP_THRESHOLD or len(pairs) < 3:
        return None, deepest_amp
    try:
        popt, _ = curve_fit(lambda z, A0, d: A0 * np.exp(-z / d), z, a, p0=[a[0], 10.0], maxfev=5000)
        A0, d = popt
        if A0 <= 0 or d <= 0:
            return None, deepest_amp
        return round(-d * np.log(ZAA_SIGNAL / A0), 1), deepest_amp
    except Exception:
        return None, deepest_amp


zaa_depths = {}
for code in ["AH", "CJ", "HS", "SR", "GT", "CV"]:
    pairs = sensor_amplitudes(code)
    zaa, deepest = fit_zaa(pairs)
    zaa_depths[code] = zaa
    was = PAPER_ZAA[code]
    now = f"{zaa} m" if zaa else f"skipped (deepest amplitude {deepest:.2f} °C)"
    print(f"{code}: fig03 {was if was else '—':>4}  ->  {now}")
    print(f"     depth/amplitude: {[(round(p[0], 1), round(p[1], 3)) for p in pairs]}")

### 4.2 Build the figure

In [ ]:
lower_y   = -5.2
base_fs   = 18
dpi       = 300
line_w    = 3.2
ann_fs    = base_fs - 5
panel_lbl = {"AH": "a", "CJ": "b", "HS": "c", "SR": "d", "GT": "e", "CV": "f"}
glacier   = {"AH": "Alphubel", "CJ": "Chessjen", "HS": "Hohsaas",
             "SR": "Sex Rouge", "GT": "Tortin", "CV": "Corvatsch"}

# Shared time axis: earliest post-equilibration date to just past the last read-out
x_start = min(pd.to_datetime(d) for d in deploy.values()) + pd.Timedelta(days=equil_days)
x_end   = pd.Timestamp("2026-08-05")

# Colormap over all plotted initial sensor depths
all_depths = []
for code in top + bot:
    di = depths_initial[code]
    for bi, lab in enumerate(labels[code]):
        if excl.get(code) and lab in excl[code]:
            continue
        all_depths += [di[bi * 2], di[bi * 2 + 1]]
cmap = mcolors.LinearSegmentedColormap.from_list(
    "magma_r_trunc", plt.get_cmap("magma_r")(np.linspace(0.15, 1.0, 256)))
norm = mcolors.Normalize(vmin=min(all_depths), vmax=max(all_depths))
depth_color = lambda d: cmap(norm(d))

fig = plt.figure(figsize=(15.5, 11.5), dpi=dpi)
gs = GridSpec(3, 3, figure=fig, height_ratios=[0.08, 1, 1],
              wspace=0.05, hspace=0.20, left=0.07, right=0.98, top=0.91, bottom=0.06)

for ri, row in enumerate([top, bot]):
    shared_y = None
    for ci, code in enumerate(row):
        ax = fig.add_subplot(gs[ri + 1, ci], sharey=shared_y)
        shared_y = shared_y or ax

        dfs    = load_bh_data(files[code], smooth_days)
        d_curr = list(depths_current[code])
        d_init = list(depths_initial[code])
        leg_entries = []

        # On a shared axis each panel starts before its own deployment, so the
        # drilling-equilibration phase is masked per site instead of by xlim.
        cutoff = np.datetime64(pd.to_datetime(deploy[code]) + pd.Timedelta(days=equil_days))

        for bi, lab in enumerate(labels[code]):
            if excl.get(code) and lab in excl[code]:
                continue
            df = dfs[bi] if bi < len(dfs) else pd.DataFrame()
            if df.empty:
                continue

            ls = "-" if lab.endswith("1TT") else "--"
            if code == "GT" and lab == "GT1TT":
                wd_curr, bd_curr = 1.4, 6.4
            else:
                wd_curr, bd_curr = d_curr[bi * 2], d_curr[bi * 2 + 1]
            wd_init, bd_init = d_init[bi * 2], d_init[bi * 2 + 1]

            fi_w, fi_b = first_nan(df["White Probe Temperature"]), first_nan(df["Black Probe Temperature"])
            skip_black = False
            if lab == "CV2TT":                     # 9.3 m probe failed at deployment
                fi_b, fi_w, skip_black = 1250, 1250, True

            t  = df["TIME"].values
            wp = df["White Probe Temperature"].values
            bp = df["Black Probe Temperature"].values
            i0 = int(np.searchsorted(t, cutoff))   # failure indices stay relative to the full frame

            ax.plot(t[i0:fi_w], wp[i0:fi_w], ls=ls, color=depth_color(wd_init), lw=line_w)
            if fi_w < len(t):
                s = max(fi_w, i0)
                ax.plot(t[s:], wp[s:], ls=":", color=depth_color(wd_init), lw=line_w)
            leg_entries.append((ls, depth_color(wd_init), wd_init, wd_curr))

            if bd_init is not None and not skip_black:
                ax.plot(t[i0:fi_b], bp[i0:fi_b], ls=ls, color=depth_color(bd_init), lw=line_w)
                if fi_b < len(t):
                    s = max(fi_b, i0)
                    ax.plot(t[s:], bp[s:], ls=":", color=depth_color(bd_init), lw=line_w)
                leg_entries.append((ls, depth_color(bd_init), bd_init, bd_curr))

        if code == "CJ":                           # summer 2025 refreezing event
            for bi, lab in enumerate(labels[code]):
                if lab.endswith("2TT") and np.isclose(d_init[bi * 2], 8.5, atol=0.3) and not dfs[bi].empty:
                    d2 = dfs[bi]
                    k = int(np.argmin(np.abs(d2["TIME"] - pd.Timestamp("2025-07-20"))))
                    ax.annotate("refreezing\nmeltwater",
                                xy=(d2["TIME"].values[k], d2["White Probe Temperature"].values[k]),
                                xytext=(d2["TIME"].values[k] + np.timedelta64(35, "D"), 0.35),
                                arrowprops=dict(arrowstyle="->", color="gray", lw=1.5),
                                fontsize=base_fs - 7, color="gray", ha="left", va="top")
                    break

        ax.axvline(pd.to_datetime(deploy[code]), color="gray", lw=1.5, alpha=0.8, zorder=0)
        ax.axhline(0, color="k", lw=0.8, ls="--", alpha=0.5)
        ax.set_ylim(lower_y, 0.5)
        ax.set_xlim(x_start, x_end)
        loc = mdates.AutoDateLocator(minticks=5, maxticks=7)
        ax.xaxis.set_major_locator(loc)
        ax.xaxis.set_major_formatter(mdates.ConciseDateFormatter(loc))
        ax.tick_params(axis="both", labelsize=base_fs - 4)
        ax.grid(True, which="major", color="lightgrey", linewidth=0.6, zorder=0)

        if ci > 0:
            ax.set_ylabel("")
            ax.tick_params(labelleft=False)
        else:
            ax.set_ylabel("Temperature [°C]", fontsize=base_fs - 2)

        ax.text(0.02, 0.02, f"{glacier[code]} ({code})", transform=ax.transAxes,
                ha="left", va="bottom", fontsize=base_fs - 3, weight="bold",
                bbox=dict(boxstyle="round,pad=0.15", fc="white", ec="k", alpha=0.7))
        ax.text(0.02, 0.98, f"({panel_lbl[code]})", transform=ax.transAxes,
                ha="left", va="top", fontsize=base_fs, weight="bold",
                bbox=dict(boxstyle="round,pad=0.15", fc="white", ec="none", alpha=1.0))

        if zaa_depths.get(code) is not None:
            ax.text(0.02, 0.09, f"ZAA ≈ {zaa_depths[code]:.0f} m", transform=ax.transAxes,
                    ha="left", va="bottom", fontsize=ann_fs, color="black", style="italic",
                    bbox=dict(boxstyle="round,pad=0.15", fc="white", ec="none", alpha=0.85))

        if leg_entries:
            handles = [Line2D([0], [0], color=c, lw=line_w, ls=ls) for ls, c, _, _ in leg_entries]
            texts   = [f"{di:.1f}→{dc:.1f} m" for _, _, di, dc in leg_entries]
            ax.legend(handles, texts, loc="upper center" if code == "CV" else "lower right",
                      fontsize=ann_fs, frameon=True, fancybox=False, edgecolor="black",
                      framealpha=0.92, facecolor="white", handlelength=1.6, handleheight=0.8,
                      borderpad=0.4, labelspacing=0.25,
                      title="$Z_{init}\\;\\rightarrow\\;Z_{final}$", title_fontsize=ann_fs)

# ── Legend bar ───────────────────────────────────────────────────────────────
fl, fb, fw, fh = 0.07, 0.862, 0.91, 0.124
frame_ax = fig.add_axes([fl, fb, fw, fh])
frame_ax.set_xlim(0, 1); frame_ax.set_ylim(0, 1)
frame_ax.patch.set_facecolor("white")
frame_ax.tick_params(left=False, bottom=False, labelleft=False, labelbottom=False)
for spine in frame_ax.spines.values():
    spine.set_visible(True); spine.set_linewidth(1.0); spine.set_color("black")
frame_ax.text(0.007, 0.93, "Legend", transform=frame_ax.transAxes,
              ha="left", va="top", fontsize=base_fs - 3, fontweight="bold")

cbar_ax = fig.add_axes([fl + 0.03, fb + 0.058, 0.51, 0.020])
cb = ColorbarBase(cbar_ax, cmap=cmap, norm=norm, orientation="horizontal")
cb.set_label("Initial sensor depth [m]", fontsize=base_fs - 4, labelpad=2)
cb.ax.tick_params(labelsize=base_fs - 3)

leg_ax = fig.add_axes([fl + 0.60, fb - 0.002, 0.30, fh + 0.004])
leg_ax.set_axis_off()
leg_ax.legend(handles=[Line2D([0], [0], color="k", lw=line_w, ls="-",  label="1TT"),
                       Line2D([0], [0], color="k", lw=line_w, ls="--", label="2TT")],
              loc="center", ncol=2, frameon=False, fontsize=base_fs - 2,
              title="Borehole", title_fontsize=base_fs - 4)

fig.supxlabel("Time", fontsize=base_fs, y=0.01)
savefig(fig, OUT_DIR + "tt_timeseries_all6_depth_colored_extended.pdf", dpi=dpi, bbox_inches="tight")
plt.show()

## 5. Geoprecision heatmap mosaics (extended figS14–S16)

Same construction, colour scale and post-equilibration start date as figS14–S16; only the end
date moves to the 2026 read-out. Keeping `cbar_min` unchanged keeps the panels directly
comparable with the published versions — where a full year of data now exceeds the scale
(most visibly AH2G, which reaches −6.3 °C in February) the near-surface saturates in the
lowest bin, but the magnitude stays readable in the time-series row below.

Two things a full year exposes that a one-month window did not:

- **Logger faults.** Several sensors report a −99 °C fault code or drop out entirely — most
  importantly HS2G, whose two deepest sensors (16.5 m, 21.5 m) die on 16 Jan 2026 and whose
  remaining sensors record only intermittently from mid-February. The heatmap builder fills
  gaps by interpolating in time, which would silently carry a dead sensor forward at its last
  value for months, so faults are masked and the affected part of the grid is left blank.
- **Borehole thermal recovery.** The published one-month windows start ~3 weeks after
  installation, so they still contain drilling heat — the large temperate (dark red) zone in
  AH1G below 22 m relaxes away over the following months.

In [ ]:
TIME_FREQ = "3h"
SENTINEL_LO, SENTINEL_HI = -20.0, 10.0   # outside this range the reading is a logger fault


class CleanChain(ThermistorData):
    """ThermistorData that drops logger fault codes and remembers what it returned."""

    def get_chain_data_with_offsets(self, *args, **kwargs):
        df = super().get_chain_data_with_offsets(*args, **kwargs)
        cols = [c for c in df.columns if c.startswith("#")]
        df[cols] = df[cols].mask((df[cols] < SENTINEL_LO) | (df[cols] > SENTINEL_HI))
        self.last_clean_df = df.copy()
        return df


def blank_missing(meta, clean_df, freq=TIME_FREQ):
    """NaN out grid cells and series points that have no underlying measurement."""
    sensor_depths = meta["sensor_depths"]
    cols = [c for c in clean_df.columns if c in sensor_depths]
    have = ((clean_df.set_index("TIME")[cols].resample(freq).count()
             .reindex(meta["time_index"]).fillna(0)).to_numpy() > 0)
    depths = np.array([sensor_depths[c] for c in cols], float)

    raw = meta["raw_df"]                       # series row: gaps instead of interpolation
    for j, c in enumerate(cols):
        raw.loc[~have[:, j], c] = np.nan

    zg, grid = meta["depth_grid"], meta["grid"]
    for i in range(grid.shape[0]):             # heatmap: only the span bracketed by live sensors
        live = depths[have[i, :]]
        if live.size == 0:
            grid[i, :] = np.nan
        else:
            grid[i, (zg < live.min()) | (zg > live.max())] = np.nan
    return meta, int(np.isnan(grid).sum())


# borehole -> (chain, spliced file, depth settings, bedrock depth [m])
CHAINS = {
    "AH1G": ("A551FE", "A551FE_20250714_20260730_spliced.csv", "thermistor_settings_ah1g.csv", 50.6),
    "AH2G": ("A55204", "A55204_20250716_20260730_spliced.csv", "thermistor_settings_ah2g.csv", 20.3),
    "AH3G": ("A55205", "A55205_20250716_20260721_spliced.csv", "thermistor_settings_ah3g.csv", 58.3),
    "HS1G": ("A551FD", "A551FD_20250714_20260729_spliced.csv", "thermistor_settings_hs1g.csv", 29.0),
    "HS2G": ("A55203", "A55203_20250716_20260729_spliced.csv", "thermistor_settings_hs2g.csv", 21.5),
    "HS3G": ("A55200", "A55200_20250715_20260729_spliced.csv", "thermistor_settings_hs3g.csv", 31.0),
    "CJ1G": ("A55201", "A55201_20250715_20260710_spliced.csv", "thermistor_settings_cj1g.csv", 38.3),
    "CJ2G": ("A55202", "A55202_20250715_20260710_spliced.csv", "thermistor_settings_cj2g.csv", 17.0),
}

# site -> (start, end, cbar_min) — start and cbar_min as in figS14-S16
WINDOWS = {
    "AH": ("07.08.2025 10:00:00", "30.07.2026 08:00:00", -3.0),
    "HS": ("15.08.2025 12:00:00", "29.07.2026 09:00:00", -1.5),
    "CJ": ("08.08.2025 12:00:00", "10.07.2026 09:00:00", -2.0),
}


def build_heatmap(bh, site):
    chain, fname, depth_csv, bedrock = CHAINS[bh]
    start, end, cbar_min = WINDOWS[site]
    td = CleanChain(GP_FULL + fname, ",", BH_SET + depth_csv)
    fig, ax, meta = plot_chain_temperature_heatmap(
        td, start_time=start, end_time=end,
        offsets=OFFSETS_GP.loc[chain].dropna().to_dict(),
        depth_file=BH_SET + depth_csv,
        time_freq=TIME_FREQ, depth_step=0.01,
        smooth_time_sigma=1.0, smooth_depth_sigma=0.5,
        temp_step=0.2, cbar_min=cbar_min, bedrock_depth=bedrock,
    )
    plt.close(fig)
    meta, n_blank = blank_missing(meta, td.last_clean_df)
    g = meta["grid"]
    print(f"  {bh}: {np.nanmin(g):6.2f} .. {np.nanmax(g):5.2f} °C   blanked grid cells: {n_blank:,}")
    return meta

### 5.1 Alphubel

In [ ]:
metas_ah = [build_heatmap(bh, "AH") for bh in ("AH1G", "AH2G", "AH3G")]
fig_ah, _ = mosaic_chain_heatmaps(
    metas_ah, titles=["AH1G", "AH2G", "AH3G"], panel_tags=("a", "b", "c"),
    figsize=(19, 10), two_rows=True, line_width=1.9, line_alpha=0.9,
    line_color_base=plt.cm.viridis,
    contour_kwargs={"colors": "black", "linewidths": 1.0, "alpha": 0.35},
    ts_y_limits=((-2.5, 0.2), (-7.0, 0.2), (-1.0, 0.2)), ts_y_tick_steps=[0.5, 1.0, 0.2],
)
savefig(fig_ah, OUT_DIR + "mosaic_AH1G_AH2G_AH3G_extended.pdf", dpi=300, bbox_inches="tight")
plt.show()

### 5.2 Chessjen

In [ ]:
metas_cj = [build_heatmap(bh, "CJ") for bh in ("CJ1G", "CJ2G")]
fig_cj, _ = mosaic_chain_heatmaps(
    metas_cj, titles=["CJ1G", "CJ2G"], panel_tags=("a", "b"),
    figsize=(12, 10), two_rows=True, line_width=1.8, line_alpha=0.9,
    line_color_base=plt.cm.viridis,
    contour_kwargs={"colors": "black", "linewidths": 1.0, "alpha": 0.35},
    ts_y_limits=((-1.5, 0.5), (-2.6, 0.5)), ts_y_tick_steps=[0.5, 0.5],
)
savefig(fig_cj, OUT_DIR + "mosaic_CJ1G_CJ2G_extended.pdf", dpi=300, bbox_inches="tight")
plt.show()

### 5.3 Hohsaas

In [ ]:
metas_hs = [build_heatmap(bh, "HS") for bh in ("HS1G", "HS2G", "HS3G")]
fig_hs, _ = mosaic_chain_heatmaps(
    metas_hs, titles=["HS1G", "HS2G", "HS3G"], panel_tags=("a", "b", "c"),
    figsize=(19, 10), two_rows=True, line_width=1.9, line_alpha=0.9,
    line_color_base=plt.cm.viridis,
    contour_kwargs={"colors": "black", "linewidths": 1.0, "alpha": 0.35},
    ts_y_limits=((-4.0, 0.2), (-2.0, 0.2), (-4.0, 0.2)), ts_y_tick_steps=[1.0, 0.5, 1.0],
)
savefig(fig_hs, OUT_DIR + "mosaic_HS1G_HS2G_HS3G_extended.pdf", dpi=300, bbox_inches="tight")
plt.show()

## 6. Check: the spliced records reproduce the published figures

Over the window used by figS14–S16, the spliced chain files must give exactly the grids the
original single read-out files gave. A non-zero difference means the splice changed history.

In [ ]:
ORIGINALS = {
    "AH1G": ("A551FE", "2025/A551FE/raw/A551FE_20250916075641.csv", "thermistor_settings_ah1g.csv", 50.6, "AH"),
    "AH2G": ("A55204", "2025/A55204/raw/A55204_20250916082002.csv", "thermistor_settings_ah2g.csv", 20.3, "AH"),
    "AH3G": ("A55205", "2025/A55205/raw/A55205_20250916074654.csv", "thermistor_settings_ah3g.csv", 58.3, "AH"),
    "HS1G": ("A551FD", "2025/A551FD/raw/A551FD_20250927153221.csv", "thermistor_settings_hs1g.csv", 29.0, "HS"),
    "HS2G": ("A55203", "2025/A55203/raw/A55203_20250927151514.csv", "thermistor_settings_hs2g.csv", 21.5, "HS"),
    "HS3G": ("A55200", "2025/A55200/raw/A55200_20250927151301.csv", "thermistor_settings_hs3g.csv", 31.0, "HS"),
    "CJ1G": ("A55201", "2025/A55201/raw/A55201_20251215143911.csv", "thermistor_settings_cj1g.csv", 38.3, "CJ"),
    "CJ2G": ("A55202", "2025/A55202/raw/A55202_20251215144715.csv", "thermistor_settings_cj2g.csv", 17.0, "CJ"),
}
PAPER_END = {"AH": "07.09.2025 12:00:00", "HS": "27.09.2025 12:00:00", "CJ": "15.12.2025 12:00:00"}


def grid_for(path, depth_csv, chain, bedrock, site, end):
    td = ThermistorData(path, ",", BH_SET + depth_csv)
    fig, _, meta = plot_chain_temperature_heatmap(
        td, start_time=WINDOWS[site][0], end_time=end,
        offsets=OFFSETS_GP.loc[chain].dropna().to_dict(), depth_file=BH_SET + depth_csv,
        time_freq=TIME_FREQ, depth_step=0.01, smooth_time_sigma=1.0, smooth_depth_sigma=0.5,
        temp_step=0.2, cbar_min=WINDOWS[site][2], bedrock_depth=bedrock)
    plt.close(fig)
    return meta["grid"]


for bh, (chain, rel, depth_csv, bedrock, site) in ORIGINALS.items():
    end = PAPER_END[site]
    a = grid_for(GP_ROOT + rel, depth_csv, chain, bedrock, site, end)
    b = grid_for(GP_FULL + CHAINS[bh][1], depth_csv, chain, bedrock, site, end)
    if a.shape != b.shape:
        print(f"{bh}: SHAPE MISMATCH {a.shape} vs {b.shape}")
    else:
        print(f"{bh}: max|difference| = {np.nanmax(np.abs(a - b)):.3e} °C")